In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# ============================================================
# 1. 데이터 입력
# ============================================================

df = pd.read_csv("/content/train.csv")

print("=" * 60)
print("원본 데이터")
print("=" * 60)

print("데이터 크기:", df.shape)
print("\n컬럼:")
print(df.columns.tolist())

print("\n데이터 앞부분:")
print(df.head())


# ============================================================
# 2. ID 제거
# ============================================================

# ID는 환자 데이터를 구분하기 위한 식별자이므로
# 머신러닝 입력 변수에서는 제외

df = df.drop(columns=["ID"])

print("\nID 제거 후 데이터 크기:", df.shape)


# ============================================================
# 3. X / y 분리
# ============================================================

X = df.drop(columns=["Cancer"])
y = df["Cancer"]

print("\n입력 변수 X:", X.shape)
print("정답 변수 y:", y.shape)


# ============================================================
# 4. 학습 데이터 / 테스트 데이터 분리
# ============================================================

# 전처리 과정에서 테스트 데이터의 정보를 사용하지 않기 위해
# 먼저 Train / Test를 분리한다.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain:", X_train.shape)
print("Test :", X_test.shape)


# ============================================================
# 5. 변수 종류 구분
# ============================================================

categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_cols = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("\n범주형 변수:")
print(categorical_cols)

print("\n수치형 변수:")
print(numeric_cols)


# ============================================================
# 6. 결측치 처리
# ============================================================

print("\n" + "=" * 60)
print("결측치 처리")
print("=" * 60)

print("\n처리 전 결측치:")
print(X_train.isnull().sum())

# -------------------------
# 수치형 변수
# 중앙값으로 결측치 처리
# -------------------------

numeric_imputer = SimpleImputer(
    strategy="median"
)

X_train[numeric_cols] = numeric_imputer.fit_transform(
    X_train[numeric_cols]
)

X_test[numeric_cols] = numeric_imputer.transform(
    X_test[numeric_cols]
)


# -------------------------
# 범주형 변수
# 최빈값으로 결측치 처리
# -------------------------

categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)

X_train[categorical_cols] = categorical_imputer.fit_transform(
    X_train[categorical_cols]
)

X_test[categorical_cols] = categorical_imputer.transform(
    X_test[categorical_cols]
)

print("\n처리 후 결측치:")
print(X_train.isnull().sum().sum())


# ============================================================
# 7. 이상치 처리
# ============================================================

print("\n" + "=" * 60)
print("이상치 처리")
print("=" * 60)

# IQR 방법 사용
#
# Q1 = 25% 지점
# Q3 = 75% 지점
# IQR = Q3 - Q1
#
# 이상치 기준:
# Q1 - 1.5 * IQR
# Q3 + 1.5 * IQR
#
# 여기서는 의료 데이터의 실제 관측값을 무조건 삭제하지 않고
# 이상치를 경계값으로 대체하는 Winsorizing 방식을 사용한다.

outlier_info = {}

for col in numeric_cols:

    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # 이상치 개수
    train_outliers = (
        (X_train[col] < lower_bound) |
        (X_train[col] > upper_bound)
    ).sum()

    test_outliers = (
        (X_test[col] < lower_bound) |
        (X_test[col] > upper_bound)
    ).sum()

    outlier_info[col] = {
        "lower": lower_bound,
        "upper": upper_bound,
        "train_outliers": train_outliers,
        "test_outliers": test_outliers
    }

    # Train에서 계산한 기준을 Test에도 동일하게 적용
    X_train[col] = X_train[col].clip(
        lower=lower_bound,
        upper=upper_bound
    )

    X_test[col] = X_test[col].clip(
        lower=lower_bound,
        upper=upper_bound
    )


# 이상치 처리 결과 출력

print("\n변수별 이상치 개수:")

for col, info in outlier_info.items():

    print(
        f"{col:20s} "
        f"Train: {info['train_outliers']:5d} "
        f"Test: {info['test_outliers']:5d}"
    )


# ============================================================
# 8. One-Hot Encoding
# ============================================================

print("\n" + "=" * 60)
print("One-Hot Encoding")
print("=" * 60)

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Train으로 encoder 학습
X_train_cat = encoder.fit_transform(
    X_train[categorical_cols]
)

# Test는 Train에서 학습한 encoder 사용
X_test_cat = encoder.transform(
    X_test[categorical_cols]
)


# 인코딩된 컬럼 이름 생성

encoded_cols = encoder.get_feature_names_out(
    categorical_cols
)

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_cols,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_cols,
    index=X_test.index
)


# 기존 범주형 변수 제거

X_train = X_train.drop(
    columns=categorical_cols
)

X_test = X_test.drop(
    columns=categorical_cols
)


# 인코딩된 변수 추가

X_train = pd.concat(
    [X_train, X_train_cat],
    axis=1
)

X_test = pd.concat(
    [X_test, X_test_cat],
    axis=1
)


print("Encoding 후 Train 크기:", X_train.shape)
print("Encoding 후 Test 크기 :", X_test.shape)


# ============================================================
# 9. Scaling
# ============================================================

print("\n" + "=" * 60)
print("Scaling")
print("=" * 60)

scaler = StandardScaler()

# Scaling은 원래 수치형 변수에 대해서만 적용
X_train[numeric_cols] = scaler.fit_transform(
    X_train[numeric_cols]
)

X_test[numeric_cols] = scaler.transform(
    X_test[numeric_cols]
)


print("\nScaling 완료")

print("\nScaling 후 수치형 변수 통계:")
print(X_train[numeric_cols].describe().loc[["mean", "std"]])


# ============================================================
# 10. Feature Engineering (선택)
# ============================================================

# 이번 데이터에서는 기본 전처리만 수행하기 위해 생략한다.
#
# 만약 Feature Engineering을 추가하고 싶다면
# 다음과 같은 방법을 사용할 수 있다.
#
# 예:
#
# X_train["T4_T3_ratio"] = (
#     X_train["T4_Result"] /
#     (X_train["T3_Result"] + 1e-8)
# )
#
# X_test["T4_T3_ratio"] = (
#     X_test["T4_Result"] /
#     (X_test["T3_Result"] + 1e-8)
# )
#
# 단, 의료 데이터에서는 새 변수가 실제로 의미가 있는지
# 확인한 후 사용하는 것이 좋다.


# ============================================================
# 11. 전처리 결과 확인
# ============================================================

print("\n" + "=" * 60)
print("최종 전처리 결과")
print("=" * 60)

print("\nTrain 데이터 크기:", X_train.shape)
print("Test 데이터 크기 :", X_test.shape)

print("\n최종 Train 데이터:")
print(X_train.head())

print("\n최종 Test 데이터:")
print(X_test.head())


# ============================================================
# 12. 전처리된 데이터 저장
# ============================================================

# Cancer를 다시 붙여서 저장

train_processed = X_train.copy()
train_processed["Cancer"] = y_train

test_processed = X_test.copy()
test_processed["Cancer"] = y_test


train_processed.to_csv(
    "train_preprocessed.csv",
    index=False
)

test_processed.to_csv(
    "test_preprocessed.csv",
    index=False
)


print("\n" + "=" * 60)
print("저장 완료")
print("=" * 60)

print("train_preprocessed.csv")
print("test_preprocessed.csv")

원본 데이터
데이터 크기: (87159, 16)

컬럼:
['ID', 'Age', 'Gender', 'Country', 'Race', 'Family_Background', 'Radiation_History', 'Iodine_Deficiency', 'Smoke', 'Weight_Risk', 'Diabetes', 'Nodule_Size', 'TSH_Result', 'T4_Result', 'T3_Result', 'Cancer']

데이터 앞부분:
            ID  Age Gender Country Race Family_Background Radiation_History  \
0  TRAIN_00000   80      M     CHN  ASN          Positive           Exposed   
1  TRAIN_00001   37      M     NGA  ASN          Positive         Unexposed   
2  TRAIN_00002   71      M     CHN  MDE          Positive         Unexposed   
3  TRAIN_00003   40      F     IND  HSP          Negative         Unexposed   
4  TRAIN_00004   53      F     CHN  CAU          Negative         Unexposed   

  Iodine_Deficiency       Smoke Weight_Risk Diabetes  Nodule_Size  TSH_Result  \
0        Sufficient  Non-Smoker   Not Obese       No     0.650355    2.784735   
1        Sufficient      Smoker       Obese       No     2.950430    0.911624   
2        Sufficient  Non-Smoker  